# Unit 1 — LunarLander-v2: Land a spacecraft with PPO

**Pass criterion:** `mean_reward − std_reward ≥ 200`

**What this notebook does**
1. Watch an untrained (random) agent → it crashes.
2. Train PPO for 1M steps (~15–25 min).
3. Watch the trained agent → a soft landing.
4. Evaluate it and push it to the Hugging Face Hub.

**No GPU needed** (small MLP; CPU is faster here).

**How to run:** `Runtime → Run all`. The Hugging Face login reads the `HF_TOKEN` Colab secret.

## 1. Setup
`swig` and `cmake` are required to build the Box2D physics engine.
`gymnasium` is pinned to **0.29.1** because newer versions removed `LunarLander-v2`, while the certification checker looks for exactly the `LunarLander-v2` tag.

In [ ]:
!apt-get install -y -qq swig cmake ffmpeg > /dev/null
!pip install -q "gymnasium[box2d]==0.29.1" "stable-baselines3==2.3.2" huggingface_sb3 imageio imageio-ffmpeg moviepy

## 2. Settings
Change the line below if your Hugging Face username is different.

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "LunarLander-v2"
MODEL_NAME = "ppo-LunarLander-v2"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

In [ ]:
import base64, imageio
import gymnasium as gym
from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

## 3. The environment
- **Observation (8 numbers):** x/y position, x/y velocity, angle, angular velocity, and whether the left / right leg touches the ground.
- **Action (4 options):** 0 = do nothing, 1 = fire left engine, 2 = fire main engine, 3 = fire right engine.
- **Reward:** + for moving towards the landing pad, − for moving away. +10 per leg contact. Small − for firing an engine. −100 for crashing, +100 for landing safely.

In [ ]:
env = gym.make(ENV_ID)
obs, _ = env.reset(seed=0)
print("Observation space:", env.observation_space.shape, "->", obs.round(2))
print("Action space     :", env.action_space)
env.close()

## 4. Video helper
Plays one episode, records it as a video and displays it inline.
With `model=None` the agent presses random buttons.

In [ ]:
def show_agent(model=None, seed=0, name="agent"):
    env = gym.make(ENV_ID, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    while True:
        if model is None:
            action = env.action_space.sample()
        else:
            action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        frames.append(env.render())
        if terminated or truncated:
            break
    env.close()
    path = f"{name}.mp4"
    imageio.mimsave(path, frames, fps=30, macro_block_size=1)
    print(f"Episode reward: {total:.1f}  ({len(frames)} steps)")
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="600" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

# Untrained agent: random buttons
show_agent(model=None, name="random_agent")

## 5. Build and train the model
- `n_envs=16`: 16 copies of the game run in parallel, so data is collected 16× faster.
- `PPO`: the algorithm we implement from scratch in Unit 8. Here we use the library version.
- `gamma=0.999`: care a lot about the future — the landing reward only arrives at the end.
- `ent_coef=0.01`: keep exploring a little.
- `n_steps / batch_size / n_epochs`: how much data is collected per update and how many passes are made over it.

The progress bar shows training status. It takes **~15–25 minutes**.

In [ ]:
train_env = make_vec_env(ENV_ID, n_envs=16)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    device="cpu",   # small MLP: CPU is faster than GPU here
    verbose=0,
)

model.learn(total_timesteps=1_000_000, progress_bar=True)
model.save(MODEL_NAME)

## 6. Evaluate
The certification checker uses `mean − std`. We compute it over 10 episodes.

In [ ]:
eval_env = Monitor(gym.make(ENV_ID, render_mode="rgb_array"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= 200 else 'NOT YET ❌ (run the optional cell below)'}")

### (Optional) If the threshold is not reached
Run this only if the score is below 200: training **continues where it left off** for another 500k steps. Then re-run step 6.

In [ ]:
# Only if score < 200
# model.learn(total_timesteps=500_000, reset_num_timesteps=False, progress_bar=True)
# model.save(MODEL_NAME)

## 7. Watch the trained agent
Same starting state (seed=0) as the random agent — compare what happens now.

In [ ]:
show_agent(model, seed=0, name="trained_agent")
show_agent(model, seed=42, name="trained_agent_2")

## 8. Push to the Hugging Face Hub
The login cell reads the `HF_TOKEN` Colab secret (a token with **Write** access from https://huggingface.co/settings/tokens) and falls back to an interactive login.

`package_to_hub` re-evaluates the model, records a replay video and creates a model card tagged `LunarLander-v2`.

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_sb3 import package_to_hub

hub_eval_env = DummyVecEnv([lambda: Monitor(gym.make(ENV_ID, render_mode="rgb_array"))])

package_to_hub(
    model=model,
    model_name=MODEL_NAME,
    model_architecture="PPO",
    env_id=ENV_ID,
    eval_env=hub_eval_env,
    repo_id=REPO_ID,
    commit_message="Upload PPO LunarLander-v2 trained agent",
)
print(f"Done: https://huggingface.co/{REPO_ID}")

## 9. Check
https://huggingface.co/spaces/ThomasSimonini/Check-my-progress-Deep-RL-Course → enter your username → **Unit 1** should be green ✅

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("LunarLander-v2", "stable-baselines3", 200)